In [6]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os

In [7]:
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")


In [9]:
class ParentState(TypedDict):

    question: str
    answer_eng: str
    answer_hin: str
    

In [10]:
subgraphLLM = ChatGoogleGenerativeAI(
    model= "gemini-2.5-flash",
    google_api_key=api_key,
)
graphLLM = ChatGoogleGenerativeAI(
    model= "gemini-2.5-flash",
    google_api_key=api_key,
)


In [11]:
def translate_text(state: ParentState):

    prompt = f"""
Translate the following text to Urdu.
Keep it natural and clear. Do not add extra content.

Text:
{state["answer_eng"]}
""".strip()
    
    translated_text = subgraphLLM.invoke(prompt).content

    return {'answer_hin': translated_text}


In [12]:
subgraph = StateGraph(ParentState)

subgraph.add_node('translate_text', translate_text)

subgraph.add_edge(START, 'translate_text')
subgraph.add_edge('translate_text', END)

subgraphWorkflow = subgraph.compile()

In [13]:
def generate_answer(state: ParentState):

    answer = graphLLM.invoke(f"You are a helpful assistant. Answer clearly.\n\nQuestion: {state['question']}").content
    return {'answer_eng': answer}

In [15]:
graph = StateGraph(ParentState)

graph.add_node("answer", generate_answer)
graph.add_node("translate", subgraphWorkflow)

graph.add_edge(START, 'answer')
graph.add_edge('answer', 'translate')
graph.add_edge('translate', END)

workflow=graph.compile()

In [16]:
workflow.invoke({'question': 'What is quantum physics'})

{'question': 'What is quantum physics',
 'answer_eng': 'Quantum physics (also known as quantum mechanics or quantum theory) is a **branch of physics that deals with the behavior of matter and energy at the smallest scales: atoms and subatomic particles.**\n\nIt\'s fundamentally different from "classical physics" (like Newton\'s laws of motion or Maxwell\'s equations of electromagnetism), which describes the world at larger, everyday scales. Classical physics works perfectly for things like planets, cars, or even tiny dust particles, but it completely breaks down when you try to apply it to individual electrons, photons, or atoms.\n\nHere are the core ideas that make quantum physics so revolutionary and, frankly, a bit mind-bending:\n\n1.  **Quantization:** Energy, momentum, angular momentum, and other properties of particles are not continuous (like a ramp) but come in discrete, indivisible packets or "chunks" called **quanta**. For example, light isn\'t just a continuous wave; it\'s a